# Saadi pre-release check on a GPU

Run this before publishing a version. It installs the candidate package into the runtime, prints the library versions it resolved, checks the files bundled in the wheel, loads a Qwen model through the vLLM backend (the backend that produced the paper's results), checks that the model follows the prompt contract, and runs the whole indexing, retrieval, answering, appending, and reloading path on the bundled sample document.

**Before running:** choose *Runtime → Change runtime type → T4 GPU*. The default model (`Qwen/Qwen3-4B`, about 8 GB in half precision) fits a T4. On a larger GPU, set `MODEL` to the paper's `Qwen/Qwen3-30B-A3B-Instruct-2507`.

Every check ends in an `assert`, so the notebook stops at the first problem. The last cell prints a summary.

In [ ]:
# What to install. For a TestPyPI rehearsal, use the second form with the version under test.
SAADI_INSTALL = "saadi[vllm]"
# SAADI_INSTALL = "--index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple 'saadi[vllm]==0.1.1'"

# The model to load. "mock" runs the notebook on a CPU without any model, to check the notebook itself.
MODEL = "Qwen/Qwen3-4B"
DTYPE = "half"  # "bfloat16" on Ampere or newer; a T4 needs "half"
MAX_MODEL_LEN = 8192
GPU_MEMORY_UTILIZATION = 0.85
TENSOR_PARALLEL_SIZE = 1

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
%pip install -q {SAADI_INSTALL}

## Versions

The package must import, report its version on the command line, and sit on supported library versions. `torch` must see the GPU.

In [ ]:
import importlib.metadata as md
import platform
import sys

import saadi

print("python", platform.python_version())
print("saadi", saadi.__version__)
for name in ("numpy", "PyYAML", "tqdm", "torch", "transformers", "vllm"):
    try:
        print(f"{name:13s}", md.version(name))
    except md.PackageNotFoundError:
        print(f"{name:13s}", "not installed")

assert sys.version_info >= (3, 11), "Saadi needs Python 3.11 or newer"
if MODEL != "mock":
    import torch

    print("cuda", torch.version.cuda, "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
    assert torch.cuda.is_available(), "No GPU visible to torch; choose a GPU runtime"
    assert tuple(int(p) for p in md.version("transformers").split(".")[:2]) >= (4, 51), "transformers is older than 4.51"
!saadi --version

## Packaging

The wheel must carry the sample document and the prompt profiles, and the mock demo must run from the installed package alone.

In [ ]:
import importlib.resources as resources

package = resources.files("saadi")
for relative in ("data/aurora_report.txt", "prompts/profiles/default", "prompts/profiles/conversation/sample.txt"):
    assert package.joinpath(relative).exists(), f"missing from the installed package: {relative}"
    print("ok  ", relative)

sample_text = package.joinpath("data/aurora_report.txt").read_text(encoding="utf-8")
with open("aurora_report.txt", "w", encoding="utf-8") as handle:
    handle.write(sample_text)
print(len(sample_text.split()), "words in the sample document")

In [ ]:
!saadi demo --llm mock --state saadi_state/demo_mock --chunk-size 100

## Load the model through vLLM

The backend records what it actually loaded: the resolved model name, the Hugging Face commit, the dtype, and the vLLM version. These are the values written into every run manifest.

In [ ]:
from saadi import resolve_backend

if MODEL == "mock":
    llm = "mock"
else:
    llm = {
        "backend": "vllm",
        "model_id": MODEL,
        "dtype": DTYPE,
        "max_model_len": MAX_MODEL_LEN,
        "gpu_memory_utilization": GPU_MEMORY_UTILIZATION,
        "tensor_parallel_size": TENSOR_PARALLEL_SIZE,
        "enforce_eager": True,
    }
backend = resolve_backend(llm)
runtime = dict(getattr(backend, "runtime_metadata", {}) or {})
for key, value in runtime.items():
    print(f"{key:22s}", value)
if MODEL != "mock":
    assert runtime["backend"] == "vllm" and runtime["model_name"] == MODEL
    assert runtime.get("vllm_version"), "the backend did not record the vLLM version"

## Prompt contract

One entity-extraction prompt and one fold prompt are sent to the loaded model. The check passes when both answers parse: a JSON object with active entities and verbatim anchors ending in `[END_JSON]`, and a summary ending in `[END_SUMMARY]`.

In [ ]:
from saadi.prompts import check_prompt_profile, format_report

report = check_prompt_profile("default", backend=backend, text=" ".join(sample_text.split()[:120]))
print(format_report(report))
assert report.ok, "the model does not follow the prompt contract; see the report above"

## Index once, then retrieve, answer, append, and reload

Small chunks make the short sample fold into a hierarchy. The assertions mirror the package's own CLI test, so a pass here means the real model reproduces what the mock path guarantees.

In [ ]:
from saadi import Saadi

STATE = "saadi_state/release_check"
doc = Saadi(llm=backend, state_dir=STATE, chunk_size=100)
index_report = doc.index("aurora_report.txt")
print(index_report)
assert index_report.chunks > 5 and index_report.model_calls > 0
if MODEL != "mock":
    assert index_report.entities > 0, "the model extracted no entities"
    assert index_report.rejected_entities < index_report.entities, "most proposals had anchors that were not in the chunk"
print(len(doc.history.get(1, [])), "level-1 summaries")

In [ ]:
for entity in list(doc.memory)[:8]:
    print(entity.name, list(entity.aliases), "|", "; ".join(entity.descriptions)[:140])

In [ ]:
question = "Who keeps the Corvin station?"
hits = doc.retrieve(question, k=3)
for hit in hits:
    print(hit.rank, f"chunk {hit.chunk_index}", round(hit.score or 0.0, 3), hit.text[:100], "...")
assert any("Corvin" in hit.text for hit in hits), "the Corvin chunk is not in the top 3"

reply = doc.answer(question)
print()
print("Answer:", reply.text)
assert reply.text.strip(), "empty answer"

In [ ]:
more = (
    "In November Lake Dorn froze two weeks early. Priya Nair extended her inflow model to it over the winter, "
    "and Tomas Egger scouted a path to the new station that stays safe in snow."
)
update = doc.append(more)
print(update.chunks, "chunks,", update.new_chunks, "new,", update.model_calls, "model calls")
assert update.new_chunks >= 1 and update.chunks > index_report.chunks

again = Saadi.load(STATE)
print(again)
first = again.retrieve("Who leads the Aurora team?", k=1)[0]
print(first.text[:120], "...")
assert len(again.chunks) == update.chunks

## Summary

In [ ]:
summary = {
    "saadi": saadi.__version__,
    "model": runtime.get("model_name", MODEL),
    "hf_commit": runtime.get("hf_commit_hash"),
    "dtype": runtime.get("dtype"),
    "vllm": runtime.get("vllm_version"),
    "prompt_contract": "ok" if report.ok else "failed",
    "chunks": update.chunks,
    "entities": len(list(again.memory)),
    "model_calls": index_report.model_calls + update.model_calls,
}
for key, value in summary.items():
    print(f"{key:16s}", value)
print()
print("All checks passed.")

## Notes

- On a server with the paper's model, set `MODEL = "Qwen/Qwen3-30B-A3B-Instruct-2507"`, `DTYPE = "bfloat16"`, and `TENSOR_PARALLEL_SIZE` to the number of GPUs. The runtime metadata printed above is what a run manifest records; compare `hf_commit` with the reference manifest when reproducing a saved run.
- `MODEL = "mock"` runs the notebook on a CPU in under a minute. It checks the installation and the notebook itself, not a model.
- If the prompt-contract check fails, a larger model or a prompt profile adapted to it is the fix; small models often return anchors that are not verbatim in the chunk, and those proposals are dropped.